# Flash Flash Revolution Silver Layer Temporal Pattern Features

This notebook calculates **note-level temporal/rhythm complexity features** that capture timing and spacing patterns. These features address a critical gap - the absence of rhythm complexity information beyond basic density metrics.

## Features Computed

### 1. note_spacing (double)
**Hypothesis:** Time gaps between notes affect difficulty
- **Calculation:** Current note_id - Previous note_id
- **Value 0**: First note in song
- **Higher values**: Larger gaps between notes (potential for rhythm breaks)
- **Lower values**: Rapid-fire note sequences
- **Why it matters**: Inconsistent spacing requires rhythm adaptation

### 2. is_burst_note (Binary: 0 or 1)
**Hypothesis:** Notes in rapid bursts are harder than isolated notes
- **Value 1**: Note spacing ≤ 3 (very close to previous note)
- **Value 0**: Normal or large spacing
- **Threshold rationale**: note_id spacing of 3 or less indicates rapid succession
- **Why it matters**: Bursts require sustained speed and accuracy

### 3. is_isolated_note (Binary: 0 or 1)
**Hypothesis:** Isolated notes after gaps require rhythm reset
- **Value 1**: Note spacing ≥ 20 (significant gap before this note)
- **Value 0**: Normal spacing
- **Threshold rationale**: Large gaps break flow and require re-entry
- **Why it matters**: Rhythm breaks increase cognitive load

---

## Incremental Processing

**Change Detection:**
- Monitors `swf_version` from bronze__songlist to detect song updates
- Only processes songs that are new or have changed
- Uses DELETE + INSERT pattern for incremental updates

**Dependencies:**
- `acubed.ffr.silver__zero-framer` (decomposed notes)
- `acubed.ffr.bronze__songlist` (swf_version for change detection)

---

## Output Table

**Table:** `acubed.ffr.silver__temporal-patterns`

**Schema:**
- `song_id` (bigint): Song identifier
- `note_id` (int): Note sequence number
- `orientation` (string): Arrow pattern
- `note_spacing` (double): Gap from previous note (note_id difference)
- `is_burst_note` (int): 1 if in rapid succession
- `is_isolated_note` (int): 1 if after significant gap
- `swf_version` (bigint): Version tracking

**Granularity:** One row per note

In [0]:
from pyspark.sql.functions import (
    col, lag, when, coalesce, lit
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("✓ Imports loaded successfully")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.`silver__temporal-patterns`"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

silver_table = "acubed.ffr.`silver__temporal-patterns`"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs and updated songs
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_temporal = spark.table(silver_table).select(
        col("song_id").alias("silver_song_id"),
        col("swf_version").alias("silver_swf_version")
    ).distinct()
    
    # Left join to find new songs and updated songs
    changed_songs = bronze_songlist.join(
        silver_temporal,
        bronze_songlist.song_id == silver_temporal.silver_song_id,
        "left"
    ).filter(
        # New songs (not in silver) OR updated songs (different swf_version)
        col("silver_song_id").isNull() |
        (col("bronze_swf_version") != col("silver_swf_version"))
    )
    
    changed_song_ids = changed_songs.select(col("song_id")).distinct()
    num_changed = changed_song_ids.count()
    
    if num_changed == 0:
        print("ℹ No changed songs detected")
        changed_song_ids = None
    else:
        print(f"✓ Found {num_changed} songs to process (new or updated)")
else:
    # Force full refresh or table doesn't exist
    print("✓ Will process all songs")
    changed_song_ids = None

In [0]:
# Guard: Skip if no songs to process
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping temporal pattern calculation")
    print("✓ Notebook will complete successfully (idempotent run)")
    # Set empty DataFrame
    df_temporal = spark.createDataFrame([], schema="""
        song_id bigint, note_id int, orientation string,
        note_spacing double, is_burst_note int, is_isolated_note int,
        swf_version bigint
    """)
else:
    print("="*60)
    print("COMPUTING TEMPORAL PATTERN FEATURES")
    print("="*60)
    
    # Load decomposed notes from silver__notes-adjusted and join with swf_version
    df_notes = spark.table("acubed.ffr.`silver__notes-adjusted`").alias("n").join(
        spark.table("acubed.ffr.bronze__songlist").select(
            col("id").alias("songlist_id"),
            col("swf_version")
        ).alias("s"),
        col("n.song_id") == col("s.songlist_id"),
        "inner"
    ).select(
        col("n.song_id"),
        col("n.note_id"),
        col("n.orientation"),
        col("s.swf_version")
    )
    
    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_notes = df_notes.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Processing {df_notes.select('song_id').distinct().count()} changed songs")
    else:
        print(f"ℹ Full refresh mode: Processing all songs")
    
    # Define window for lag operation (previous note in same song)
    window_spec = Window.partitionBy("song_id").orderBy("note_id")
    
    # Calculate features
    df_temporal = df_notes.withColumn(
        "prev_note_id",
        lag(col("note_id")).over(window_spec)
    ).withColumn(
        # Feature 1: Note spacing (gap from previous note)
        "note_spacing",
        when(
            col("prev_note_id").isNotNull(),
            col("note_id") - col("prev_note_id")
        ).otherwise(lit(0.0))
    ).withColumn(
        # Feature 2: Burst note (very close spacing)
        "is_burst_note",
        when(
            (col("note_spacing") > 0) & (col("note_spacing") <= 3),
            1
        ).otherwise(0)
    ).withColumn(
        # Feature 3: Isolated note (large gap before it)
        "is_isolated_note",
        when(col("note_spacing") >= 20, 1).otherwise(0)
    ).select(
        "song_id", "note_id", "orientation",
        "note_spacing", "is_burst_note", "is_isolated_note",
        "swf_version"
    )
    
    row_count = df_temporal.count()
    print(f"\n✓ Calculated temporal patterns for {row_count:,} notes")
    print("\nFeatures computed:")
    print("  1. note_spacing - time gap from previous note")
    print("  2. is_burst_note - rapid succession indicator")
    print("  3. is_isolated_note - rhythm break indicator")

In [0]:
# Save to Delta table using DELETE + INSERT pattern for incremental updates
table_name = "acubed.ffr.`silver__temporal-patterns`"

# Capture counts BEFORE operations
rows_to_insert = df_temporal.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update")
    print("✓ Table remains unchanged (idempotent run)")
else:
    print("\n" + "="*60)
    print("SAVING TO DELTA TABLE")
    print("="*60)
    
    if spark.catalog.tableExists(table_name):
        # Table exists - use DELETE + INSERT for incremental updates
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_temporal.select("song_id").distinct()
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_temporal.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {rows_to_insert:,} new rows")
        
        # Get final count
        final_count = spark.table(table_name).count()
        print(f"\n✓ Updated {table_name}")
        print(f"  Total rows: {final_count:,}")
    else:
        # Table doesn't exist - create it
        df_temporal.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"✓ Created {table_name} with {rows_to_insert:,} rows")
    
    print("\n" + "="*60)
    print("✓ Temporal pattern features ready!")
    print("="*60)

In [0]:
%sql
SELECT 
  song_id,
  COUNT(*) as total_notes,
  ROUND(AVG(note_spacing), 2) as avg_spacing,
  ROUND(STDDEV(note_spacing), 2) as spacing_std,
  SUM(is_burst_note) as burst_notes,
  SUM(is_isolated_note) as isolated_notes,
  ROUND(SUM(is_burst_note) * 100.0 / COUNT(*), 1) as pct_burst
FROM acubed.ffr.`silver__temporal-patterns`
GROUP BY song_id
ORDER BY total_notes DESC
LIMIT 20